# Forecasting Model: Predicting Next Month's Delay Rate

This notebook adapts the nowcasting model (notebook 8b) to a **forecasting** model that predicts next month's delay rate using only data available **before** the target month. One obvious challenge is the unavailability of target month's weather data, because weather data are published only after the month ends. The nowcasting model uses same-month weather features (11.2% combined importance):
- `rainy_days_arr_exp` (5.5%)
- `temp_volatility_total_exp` (3.2%)
- `extreme_weather_days_total` (2.5%)

Three strategies to solve this problem are tested in this notebook:
1. **No Weather**: Remove weather features entirely (baseline)
2. **Lag1 Weather**: Use previous month's weather
3. **Climatology**: Use historical monthly averages

## Reference: Nowcasting Performance (notebook 8b)

| Model | Test R2 | Test F1 |
|-------|---------|--------|
| Ridge | 0.462 | - |
| Random Forest | 0.486 | 0.696 |
| XGBoost | - | 0.733 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    f1_score, roc_auc_score, precision_score, recall_score
)
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

%matplotlib inline

## 1. Data Preparation

Load the same data as notebook 8b and apply identical filtering.

In [ ]:
# Load multi-route data
df = pd.read_csv('../data/processed/ml_training_data_multiroute_hols.csv')

df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print(f"Original shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

In [ ]:
# Filter low-volume airline-routes (same as 8b: 40 flights/month threshold)
volume_threshold = 40
airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean()
high_volume_ar = airline_route_volume[airline_route_volume >= volume_threshold].index.tolist()

df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()
print(f"Records after volume filtering: {len(df_filtered)}")

# Exclude anomalous routes (same as 8b)
anomalous_routes = ['Melbourne_Hobart', 'Adelaide_Sydney', 'Perth_Brisbane']
df_filtered = df_filtered[~df_filtered['route'].isin(anomalous_routes)].copy()
print(f"Records after route filtering: {len(df_filtered)}")

## 2. Feature Engineering

Create all feature variants needed for the ablation study.

In [ ]:
# Lagged delay rate features (same as 8b)
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']

# Cyclical month encoding
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encoding
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = list(route_dummies.columns)

print(f"Airline columns: {len(airline_cols)}")
print(f"Route columns: {len(route_cols)}")

In [ ]:
# Weather feature variants

# 1. Same-month weather (for reference, used in nowcasting)
df_filtered['temp_volatility_total'] = df_filtered['temp_volatility_dep'] + df_filtered['temp_volatility_arr']
df_filtered['extreme_weather_days_total'] = df_filtered['extreme_weather_days_dep'] + df_filtered['extreme_weather_days_arr']

# Store max values for exp transforms (computed on training data later)
rainy_max = df_filtered['rainy_days_arr'].max()
temp_vol_max = df_filtered['temp_volatility_total'].max()

# Same-month exp transforms
df_filtered['rainy_days_arr_exp'] = np.exp(df_filtered['rainy_days_arr'] / rainy_max)
df_filtered['temp_volatility_total_exp'] = np.exp(df_filtered['temp_volatility_total'] / temp_vol_max)

# 2. Lag1 weather (previous month)
df_filtered['rainy_days_arr_lag1'] = df_filtered.groupby('airline_route')['rainy_days_arr'].shift(1)
df_filtered['temp_volatility_total_lag1'] = df_filtered.groupby('airline_route')['temp_volatility_total'].shift(1)
df_filtered['extreme_weather_days_total_lag1'] = df_filtered.groupby('airline_route')['extreme_weather_days_total'].shift(1)

# Lag1 exp transforms
df_filtered['rainy_days_arr_lag1_exp'] = np.exp(df_filtered['rainy_days_arr_lag1'] / rainy_max)
df_filtered['temp_volatility_total_lag1_exp'] = np.exp(df_filtered['temp_volatility_total_lag1'] / temp_vol_max)

# 3. Climatology (historical monthly averages from training years only)
train_years = list(range(2010, 2018)) + [2023]
train_mask_clim = df_filtered['year'].isin(train_years)

for col in ['rainy_days_arr', 'temp_volatility_total', 'extreme_weather_days_total']:
    climatology = df_filtered[train_mask_clim].groupby(['route', 'month_num'])[col].mean()
    df_filtered[f'{col}_climatology'] = df_filtered.apply(
        lambda row: climatology.get((row['route'], row['month_num']), np.nan), axis=1
    )

# Climatology exp transforms
df_filtered['rainy_days_arr_climatology_exp'] = np.exp(df_filtered['rainy_days_arr_climatology'] / rainy_max)
df_filtered['temp_volatility_total_climatology_exp'] = np.exp(df_filtered['temp_volatility_total_climatology'] / temp_vol_max)

print("Weather feature variants created.")

In [ ]:
# Drop NaN from lag features
# Note: forecasting mode requires lag1 weather, which adds 1 extra NaN row per airline-route
df_clean = df_filtered.dropna(subset=[
    'delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient',
    'rainy_days_arr_lag1', 'temp_volatility_total_lag1', 'extreme_weather_days_total_lag1'
]).copy()

print(f"Rows after dropping NaN: {len(df_clean)}")

## 3. Train/Validation/Test Split

Same split as notebook 8b.

In [ ]:
train_mask = (((df_clean['year'] >= 2010) & (df_clean['year'] <= 2017)) | (df_clean['year'] == 2023))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2024))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print(f"Train (2010-2017, 2023): {train_mask.sum()} samples")
print(f"Validation (2018, 2024): {val_mask.sum()} samples")
print(f"Test (2019, 2025+):      {test_mask.sum()} samples")

In [ ]:
# Define feature sets for each weather strategy
base_features = airline_cols + route_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']
non_weather_features = ['delay_rate_gradient', 'n_public_holidays_total', 'pct_school_holiday']

# A1: No weather
features_no_weather = base_features + non_weather_features

# A2: Lag1 weather
weather_lag1 = ['rainy_days_arr_lag1_exp', 'temp_volatility_total_lag1_exp', 'extreme_weather_days_total_lag1']
features_lag1_weather = base_features + weather_lag1 + non_weather_features

# A3: Climatology weather
weather_climatology = ['rainy_days_arr_climatology_exp', 'temp_volatility_total_climatology_exp', 'extreme_weather_days_total_climatology']
features_climatology = base_features + weather_climatology + non_weather_features

# Reference: Nowcasting (same-month weather)
weather_nowcast = ['rainy_days_arr_exp', 'temp_volatility_total_exp', 'extreme_weather_days_total']
features_nowcast = base_features + weather_nowcast + non_weather_features

print(f"Features (No Weather):   {len(features_no_weather)}")
print(f"Features (Lag1 Weather): {len(features_lag1_weather)}")
print(f"Features (Climatology):  {len(features_climatology)}")
print(f"Features (Nowcast ref):  {len(features_nowcast)}")

## 4. Weather Strategy Ablation Study

Compare three forecasting strategies plus nowcasting reference.

In [ ]:
def prepare_data(df, features, train_mask, val_mask, test_mask):
    """Prepare train/val/test data for a given feature set."""
    X_train = df.loc[train_mask, features].values
    X_val = df.loc[val_mask, features].values
    X_test = df.loc[test_mask, features].values
    
    y_train_reg = df.loc[train_mask, 'delay_rate'].values
    y_val_reg = df.loc[val_mask, 'delay_rate'].values
    y_test_reg = df.loc[test_mask, 'delay_rate'].values
    
    y_train_clf = df.loc[train_mask, 'is_high_delay'].values
    y_val_clf = df.loc[val_mask, 'is_high_delay'].values
    y_test_clf = df.loc[test_mask, 'is_high_delay'].values
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'X_train_scaled': X_train_scaled, 'X_val_scaled': X_val_scaled, 'X_test_scaled': X_test_scaled,
        'y_train_reg': y_train_reg, 'y_val_reg': y_val_reg, 'y_test_reg': y_test_reg,
        'y_train_clf': y_train_clf, 'y_val_clf': y_val_clf, 'y_test_clf': y_test_clf,
        'scaler': scaler
    }

def train_and_evaluate(data, strategy_name):
    """Train models and return evaluation results."""
    results = {'strategy': strategy_name}
    
    # Ridge Regression
    ridge = Ridge(alpha=100)
    ridge.fit(data['X_train_scaled'], data['y_train_reg'])
    ridge_pred = ridge.predict(data['X_test_scaled'])
    results['ridge_r2'] = r2_score(data['y_test_reg'], ridge_pred)
    results['ridge_rmse'] = np.sqrt(mean_squared_error(data['y_test_reg'], ridge_pred))
    results['ridge_mae'] = mean_absolute_error(data['y_test_reg'], ridge_pred)
    
    # Random Forest Regression
    rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_reg.fit(data['X_train'], data['y_train_reg'])
    rf_pred = rf_reg.predict(data['X_test'])
    results['rf_r2'] = r2_score(data['y_test_reg'], rf_pred)
    results['rf_rmse'] = np.sqrt(mean_squared_error(data['y_test_reg'], rf_pred))
    
    # XGBoost Classification
    if HAS_XGB:
        xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                     min_child_weight=5, random_state=42, n_jobs=-1)
        xgb_clf.fit(data['X_train'], data['y_train_clf'], eval_set=[(data['X_val'], data['y_val_clf'])], verbose=False)
        xgb_pred = xgb_clf.predict(data['X_test'])
        xgb_proba = xgb_clf.predict_proba(data['X_test'])[:, 1]
        results['xgb_f1'] = f1_score(data['y_test_clf'], xgb_pred)
        results['xgb_auc'] = roc_auc_score(data['y_test_clf'], xgb_proba)
    else:
        results['xgb_f1'] = np.nan
        results['xgb_auc'] = np.nan
    
    return results, ridge, rf_reg

In [ ]:
# Run ablation study
strategies = [
    ('A1: No Weather', features_no_weather),
    ('A2: Lag1 Weather', features_lag1_weather),
    ('A3: Climatology', features_climatology),
    ('Nowcast (ref)', features_nowcast),
]

all_results = []
best_model = None
best_rf = None
best_features = None

for strategy_name, features in strategies:
    print(f"Training: {strategy_name}")
    data = prepare_data(df_clean, features, train_mask, val_mask, test_mask)
    results, ridge, rf_reg = train_and_evaluate(data, strategy_name)
    all_results.append(results)
    
    print(f"  Ridge R2:  {results['ridge_r2']:.4f}")
    print(f"  RF R2:     {results['rf_r2']:.4f}")
    if HAS_XGB:
        print(f"  XGB F1:    {results['xgb_f1']:.4f}")
    print()
    
    # Track best forecasting model (exclude nowcast reference)
    if 'Nowcast' not in strategy_name:
        if best_model is None or results['rf_r2'] > best_model['rf_r2']:
            best_model = results
            best_rf = rf_reg
            best_features = features

results_df = pd.DataFrame(all_results)

In [ ]:
# Display results comparison
print("=" * 90)
print("WEATHER STRATEGY ABLATION RESULTS")
print("=" * 90)

header = "{:<20} {:>10} {:>10} {:>10} {:>12}".format('Strategy', 'Ridge R2', 'RF R2', 'XGB F1', 'Ridge RMSE')
print(header)
print("-" * 70)

for _, row in results_df.iterrows():
    xgb_f1_str = "{:.4f}".format(row['xgb_f1']) if not np.isnan(row['xgb_f1']) else 'N/A'
    line = "{:<20} {:>10.4f} {:>10.4f} {:>10} {:>12.4f}".format(
        row['strategy'], row['ridge_r2'], row['rf_r2'], xgb_f1_str, row['ridge_rmse']
    )
    print(line)

# Calculate delta vs nowcasting
nowcast_r2 = results_df[results_df['strategy'] == 'Nowcast (ref)']['ridge_r2'].values[0]
nowcast_rf_r2 = results_df[results_df['strategy'] == 'Nowcast (ref)']['rf_r2'].values[0]

print()
print("=" * 90)
print("DELTA VS NOWCASTING")
print("=" * 90)

header2 = "{:<20} {:>12} {:>12}".format('Strategy', 'Delta Ridge', 'Delta RF')
print(header2)
print("-" * 50)

for _, row in results_df.iterrows():
    if 'Nowcast' not in row['strategy']:
        delta_ridge = row['ridge_r2'] - nowcast_r2
        delta_rf = row['rf_r2'] - nowcast_rf_r2
        line = "{:<20} {:>+12.4f} {:>+12.4f}".format(row['strategy'], delta_ridge, delta_rf)
        print(line)

In [ ]:
# Visualize ablation results
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

strategies_short = ['No Weather', 'Lag1 Weather', 'Climatology', 'Nowcast (ref)']
x = np.arange(len(strategies_short))
width = 0.35

# R2 comparison
ax = axes[0]
ridge_r2_vals = results_df['ridge_r2'].values
rf_r2_vals = results_df['rf_r2'].values

bars1 = ax.bar(x - width/2, ridge_r2_vals, width, label='Ridge', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, rf_r2_vals, width, label='Random Forest', color='darkorange', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(strategies_short, rotation=15, ha='right')
ax.set_ylabel('Test R2')
ax.set_title('Regression Performance by Weather Strategy')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 0.6)

# F1 comparison (if XGBoost available)
ax = axes[1]
if HAS_XGB:
    xgb_f1_vals = results_df['xgb_f1'].values
    colors = ['#e74c3c' if 'Nowcast' in s else '#27ae60' for s in results_df['strategy']]
    ax.bar(x, xgb_f1_vals, color=colors, alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(strategies_short, rotation=15, ha='right')
    ax.set_ylabel('Test F1')
    ax.set_title('Classification Performance (XGBoost)')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 0.8)
else:
    ax.text(0.5, 0.5, 'XGBoost not available', ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Best Forecasting Model Analysis

Detailed analysis of the best-performing forecasting strategy.

In [ ]:
print("Best Forecasting Strategy: {}".format(best_model['strategy']))
print("  Ridge R2: {:.4f}".format(best_model['ridge_r2']))
print("  RF R2:    {:.4f}".format(best_model['rf_r2']))
print("  XGB F1:   {:.4f}".format(best_model['xgb_f1']))

In [ ]:
# Feature importance for best model
importance_df = pd.DataFrame({
    'Feature': best_features,
    'Importance': best_rf.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (Best Forecasting Model - RF):")
print("-" * 55)
for i, row in importance_df.head(15).iterrows():
    print("  {:<40} {:.4f}".format(row['Feature'], row['Importance']))

In [ ]:
# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 6))

top_features = importance_df.head(15)
y_pos = np.arange(len(top_features))

ax.barh(y_pos, top_features['Importance'], color='steelblue', alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_features['Feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Top 15 Feature Importances ({})'.format(best_model['strategy']))
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 6. Route-Level Performance

Compare forecasting vs nowcasting performance by route.

In [ ]:
# Get predictions for best forecasting model
data_best = prepare_data(df_clean, best_features, train_mask, val_mask, test_mask)

ridge_best = Ridge(alpha=100)
ridge_best.fit(data_best['X_train_scaled'], data_best['y_train_reg'])
ridge_forecast_pred = ridge_best.predict(data_best['X_test_scaled'])

# Get nowcasting predictions for comparison
data_nowcast = prepare_data(df_clean, features_nowcast, train_mask, val_mask, test_mask)
ridge_nowcast = Ridge(alpha=100)
ridge_nowcast.fit(data_nowcast['X_train_scaled'], data_nowcast['y_train_reg'])
ridge_nowcast_pred = ridge_nowcast.predict(data_nowcast['X_test_scaled'])

# Add predictions to test data
df_test = df_clean[test_mask].copy()
df_test['forecast_pred'] = ridge_forecast_pred
df_test['nowcast_pred'] = ridge_nowcast_pred
df_test['naive_lag1'] = df_test['delay_rate_lag1']

# Route-level analysis
print("Performance by Route: Forecasting vs Nowcasting (Ridge)")
print("=" * 85)
header = "{:<20} {:>12} {:>12} {:>10} {:>12}".format('Route', 'Forecast R2', 'Nowcast R2', 'Delta R2', 'Lag1 R2')
print(header)
print("-" * 85)

route_comparison = []
for route in sorted(df_test['route'].unique()):
    route_data = df_test[df_test['route'] == route]
    forecast_r2 = r2_score(route_data['delay_rate'], route_data['forecast_pred'])
    nowcast_r2 = r2_score(route_data['delay_rate'], route_data['nowcast_pred'])
    lag1_r2 = r2_score(route_data['delay_rate'], route_data['naive_lag1'])
    delta = forecast_r2 - nowcast_r2
    
    line = "{:<20} {:>12.4f} {:>12.4f} {:>+10.4f} {:>12.4f}".format(route, forecast_r2, nowcast_r2, delta, lag1_r2)
    print(line)
    route_comparison.append({
        'Route': route, 'Forecast_R2': forecast_r2, 'Nowcast_R2': nowcast_r2,
        'Delta': delta, 'Lag1_R2': lag1_r2
    })

route_comp_df = pd.DataFrame(route_comparison)

# Overall
overall_forecast = r2_score(df_test['delay_rate'], df_test['forecast_pred'])
overall_nowcast = r2_score(df_test['delay_rate'], df_test['nowcast_pred'])
overall_lag1 = r2_score(df_test['delay_rate'], df_test['naive_lag1'])
print("-" * 85)
line = "{:<20} {:>12.4f} {:>12.4f} {:>+10.4f} {:>12.4f}".format('OVERALL', overall_forecast, overall_nowcast, overall_forecast - overall_nowcast, overall_lag1)
print(line)

In [ ]:
# Visualize route-level comparison
fig, ax = plt.subplots(figsize=(14, 6))

routes = route_comp_df['Route'].values
x = np.arange(len(routes))
width = 0.25

ax.bar(x - width, route_comp_df['Forecast_R2'], width, label='Forecasting', color='#27ae60', alpha=0.8)
ax.bar(x, route_comp_df['Nowcast_R2'], width, label='Nowcasting', color='#3498db', alpha=0.8)
ax.bar(x + width, route_comp_df['Lag1_R2'], width, label='Naive (lag1)', color='#95a5a6', alpha=0.8)

ax.set_xticks(x)
route_labels = [r.replace('_', ' to ') for r in routes]
ax.set_xticklabels(route_labels, rotation=45, ha='right')
ax.set_ylabel('R2')
ax.set_title('Route-Level Performance: Forecasting vs Nowcasting vs Naive Baseline')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## 7. Success Criteria Evaluation

In [ ]:
print("=" * 70)
print("SUCCESS CRITERIA EVALUATION")
print("=" * 70)

# Targets from plan
target_r2 = 0.40
target_f1 = 0.68
nowcast_r2_ref = 0.462  # From notebook 8b
nowcast_f1_ref = 0.733
lag1_baseline = 0.33

# Actual results
actual_r2 = best_model['ridge_r2']
actual_f1 = best_model['xgb_f1']

criteria = [
    ("Forecasting R2 > 0.40", actual_r2 > target_r2, "{:.4f}".format(actual_r2)),
    ("Forecasting F1 > 0.68", actual_f1 > target_f1, "{:.4f}".format(actual_f1)),
    ("R2 within 0.05 of nowcasting", abs(actual_r2 - nowcast_r2_ref) < 0.05, "|Delta| = {:.4f}".format(abs(actual_r2 - nowcast_r2_ref))),
    ("Beats naive lag1 baseline", actual_r2 > lag1_baseline, "{:.4f} > {}".format(actual_r2, lag1_baseline)),
]

print()
header = "{:<40} {:>10} {:>20}".format('Criterion', 'Status', 'Value')
print(header)
print("-" * 75)

all_passed = True
for criterion, passed, value in criteria:
    status = "PASS" if passed else "FAIL"
    symbol = "[OK]" if passed else "[X]"
    line = "{:<40} {:>10} {:>20}".format(criterion, symbol + " " + status, value)
    print(line)
    if not passed:
        all_passed = False

print()
print("=" * 70)
if all_passed:
    print("ALL CRITERIA PASSED - Forecasting model is ready for deployment.")
else:
    print("SOME CRITERIA NOT MET - Review results and consider adjustments.")

## 8. Summary and Observations

In [ ]:
print("=" * 80)
print("FORECASTING MODEL SUMMARY")
print("=" * 80)
print()
print("BEST WEATHER STRATEGY: {}".format(best_model['strategy']))
print()
print("FEATURE SET ({} features):".format(len(best_features)))
print("  - Lag: delay_rate_lag1, delay_rate_gradient")
print("  - Holidays: n_public_holidays_total, pct_school_holiday")
print("  - Encoding: {} airline dummies, {} route dummies, month_sin/cos".format(len(airline_cols), len(route_cols)))
print("  - Volume: sectors_scheduled")
print()
print("PERFORMANCE COMPARISON:")
print("                        Forecasting     Nowcasting     Delta")
print("  Ridge R2:           {:.4f}          0.4618         {:+.4f}".format(best_model['ridge_r2'], best_model['ridge_r2'] - 0.4618))
print("  RF R2:              {:.4f}          0.4858         {:+.4f}".format(best_model['rf_r2'], best_model['rf_r2'] - 0.4858))
print("  XGBoost F1:         {:.4f}          0.7328         {:+.4f}".format(best_model['xgb_f1'], best_model['xgb_f1'] - 0.7328))
print()

### Observations:
- Performance degradation is modest (3.5% R2 drop for Ridge)
- `delay_rate_lag1` remains the dominant predictor
- Forecasting model still significantly beats naive lag1 baseline
- Lagging weather features (previous month) still make a small contribution to the predictive performance
- However,the no weather strategy will be adopted moving forward. The justification is that weather related contribution in the previous should already be captured by `delay_rate_lag1`

## 9. Next Step

In an effort to make up for the lost predictive performance compared to nowcasting, the next notebook will explore adding the 12-month lagged delay rate to the forecasting model. The rationale is it might capture yearly seasonality better the monthly cyclical encoding.